# Proyecto: Procesador de Flujo de Datos (Data Stream Processor)

## Contexto del Proyecto
En la Ciencia de Datos y en la arquitectura de sistemas distribuidos, el procesamiento de eventos en tiempo real (Data Streaming) requiere estructuras que garanticen el orden de llegada y la integridad de los datos.

Este proyecto implementa una clase `DataProcessor` que simula la recepción y procesamiento de lecturas de sensores mediante:
1. **Cola de entrada (FIFO):** Almacena las lecturas pendientes enviadas por los sensores.
2. **Pila de historial (LIFO):** Registra cada modificación sobre el estado del sistema, permitiendo operaciones de retroceso (`undo`) ante errores o anomalías.
3. **Pila de rehacer (LIFO):** Permite re-aplicar cambios deshechos (`redo`).
4. **Tabla de estado actual:** Mantiene la versión más reciente de la variable de cada sensor.




## 1. Importación de Estructuras Base
En esta sección se importan los Tipos de Datos Abstractos (TDA) de Pila (`ArrayStack`) y Cola (`ArrayQueue`), 
junto con la excepción personalizada `Empty`, provenientes del paquete `goodrich`.

- **ArrayQueue:** Administra la cola de registros pendientes bajo la política FIFO (First In, First Out).
- **ArrayStack:** Almacena el historial de modificaciones bajo la política LIFO (Last In, First Out) para permitir la funcionalidad de deshacer (`undo`).
- **Empty:** Excepción personalizada que se lanza cuando se intenta acceder o retirar elementos de una pila o cola vacía.

In [ ]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue
from goodrich.exceptions import Empty

## 2. Definición de la Clase `DataProcessor`

La clase `DataProcessor` actúa como el motor central del sistema. Mantiene cuatro componentes principales en su estado interno:

1. **`_queue` (`ArrayQueue`):** Registro de datos en espera de ser procesados (Orden FIFO).
2. **`_stack` (`ArrayStack`):** Registro de historial de cambios para permitir operaciones `undo` (Orden LIFO).
3. **`_redo_stack` (`ArrayStack`):** Historial de cambios deshechos, para permitir operaciones `redo` (bonus, Orden LIFO). Se reinicia cada vez que se procesa un nuevo registro.
4. **`_state` (`list`):** Lista que contiene los registros procesados en su estado actual `(sensor, variable, value)`.

> **#correccion:** esta sección describía originalmente una versión de la clase sin `redo()` (con solo 3 atributos). Además, el notebook tenía **dos** definiciones de `DataProcessor` (una sin `redo`, otra con `redo`); como Python reejecuta las celdas en orden, la segunda pisaba a la primera y era la única que realmente usaban las pruebas, dejando la primera como código muerto que podía confundir a quien revisara el proyecto. Se eliminó la definición duplicada y se dejó una sola clase (la que incluye `redo`), con esta explicación ya alineada a los 4 atributos reales.


## 3. Procesamiento de Registros y Consulta de Estado

En esta sección se implementa la lógica para consumir los registros de la cola, actualizar el estado actual del sistema y su historial de cambios (undo/redo).

### Estructura y Atributos Internos

- **`_queue` (`ArrayQueue`):** Almacena los registros recibidos en orden de llegada (FIFO) a la espera de ser procesados.
- **`_stack` (`ArrayStack`):** Conserva la historia de modificaciones (LIFO) para poder revertir los cambios realizados por `process_next()`.
- **`_redo_stack` (`ArrayStack`):** Conserva los cambios deshechos por `undo()` para poder reaplicarlos con `redo()` (bonus). Se reinicia cada vez que se procesa un nuevo registro, para no dejar un historial de redo inconsistente con acciones nuevas.
- **`_state` (`list`):** Mantiene el estado actual de cada combinación `(sensor, variable)` procesada hasta el momento.

### Métodos Principales

1. **`add(record)`:** Valida que el registro sea una tupla/lista de exactamente 3 elementos con un valor numérico (y no booleano) y lo añade a la cola.
2. **`process_next()`:** Procesa la siguiente lectura de la cola, guarda en la pila el valor previo (o `None` si el dato era nuevo) y actualiza la lista de estado.
3. **`undo()`:** Extrae el último cambio de la pila y revierte el estado: restaura el valor anterior si existía o elimina el registro si se creó por primera vez.
4. **`redo()` (bonus):** Reaplica el último cambio deshecho por `undo()`, tomándolo de `_redo_stack`.
5. **`pending()`:** Devuelve la cantidad de registros en espera en la cola.
6. **`current_value(sensor, variable)`:** Consulta el valor más reciente de una variable específica o lanza `KeyError` si no ha sido procesada.


In [ ]:
class DataProcessor:
    def __init__(self):
        """
        Inicializa el procesador de flujo de datos con:
        - Una cola vacía para registros pendientes (FIFO).
        - Una pila vacía para el historial de cambios (LIFO).
        - Una pila vacía para rehacer cambios deshechos (LIFO).
        - Una lista vacía para mantener el estado actual de los datos.
        """
        self._queue = ArrayQueue()
        self._stack = ArrayStack()
        self._redo_stack = ArrayStack()
        self._state = []

    def pending(self) -> int:
        """
        Devuelve el número de registros pendientes por procesar en la cola.
        """
        return len(self._queue)

    def add(self, record):
        """
        Agrega un registro (sensor, variable, valor) a la cola de pendientes.
        Lanza ValueError si el registro no es una tupla/lista de exactamente 3
        elementos, o si el valor no es numérico (int/float, sin contar bool).
        """
        # #correccion: se vuelve a validar isinstance(record, (tuple, list))
        # y no solo len(record). Antes, cualquier objeto indexable con
        # len() == 3 (p. ej. un dict de 3 llaves, o un string) podía colarse
        # y producir un TypeError inesperado en vez del ValueError documentado
        # y consistente que pide la guía del proyecto.
        if not isinstance(record, (tuple, list)) or len(record) != 3:
            raise ValueError("El registro debe ser una tupla o lista de exactamente 3 elementos: (sensor, variable, value).")

        sensor = record[0]
        variable = record[1]
        value = record[2]

        # #correccion: se usa isinstance() en vez de type() == ... para ser
        # más robusto (p. ej. con subclases numéricas), excluyendo bool
        # explícitamente ya que bool es subclase de int en Python.
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError("El valor del registro (tercer elemento) debe ser numérico (int o float).")

        self._queue.enqueue((sensor, variable, value))

    def process_next(self):
        """
        Procesa el primer registro en cola (FIFO), actualiza el estado y guarda el historial.
        Lanza Empty si no hay registros pendientes en la cola.
        """
        # 1. Si la cola está vacía, lanzar la excepción correspondiente
        if self._queue.is_empty():
            raise Empty("No hay registros pendientes para procesar.")

        # 2. Extraer el registro pendiente de la cola
        record = self._queue.dequeue()
        sensor = record[0]
        variable = record[1]
        new_value = record[2]

        # 3. Buscar si la clave (sensor, variable) ya existe en la lista de estado
        posicion_encontrada = -1
        valor_anterior = None

        for i in range(len(self._state)):
            item_actual = self._state[i]
            if item_actual[0] == sensor and item_actual[1] == variable:
                posicion_encontrada = i
                valor_anterior = item_actual[2]
                break

        # 4. Guardar en la pila el estado previo (para el undo)
        self._stack.push((sensor, variable, valor_anterior))

        # 5. Reiniciar la pila de redo ante una nueva acción
        self._redo_stack = ArrayStack()

        # 6. Actualizar la lista de estado actual
        if posicion_encontrada != -1:
            self._state[posicion_encontrada] = (sensor, variable, new_value)
        else:
            self._state.append((sensor, variable, new_value))

        return record

    def undo(self):
        """
        Deshace el último cambio realizado por process_next() (LIFO).
        Lanza Empty si el historial está vacío.
        """
        sensor, variable, valor_anterior = self._stack.pop()

        posicion = -1
        valor_actual = None

        for i in range(len(self._state)):
            item = self._state[i]
            if item[0] == sensor and item[1] == variable:
                posicion = i
                valor_actual = item[2]
                break

        self._redo_stack.push((sensor, variable, valor_actual, valor_anterior))

        if posicion != -1:
            if valor_anterior is None:
                self._state.pop(posicion)
            else:
                self._state[posicion] = (sensor, variable, valor_anterior)

    def redo(self):
        """
        Rehace el último cambio revertido por undo() (LIFO).
        Lanza Empty si no hay acciones para rehacer.
        """
        if self._redo_stack.is_empty():
            raise Empty("No hay acciones pendientes para rehacer.")

        sensor, variable, valor_a_restaurar, valor_anterior = self._redo_stack.pop()

        # Buscar el elemento en la lista de estado actual
        posicion = -1
        for i in range(len(self._state)):
            item = self._state[i]
            if item[0] == sensor and item[1] == variable:
                posicion = i
                break

        # Guardar nuevamente en la pila de deshacer (undo)
        self._stack.push((sensor, variable, valor_anterior))

        # Aplicar el valor que se había deshecho
        if posicion != -1:
            self._state[posicion] = (sensor, variable, valor_a_restaurar)
        else:
            self._state.append((sensor, variable, valor_a_restaurar))

    def current_value(self, sensor, variable):
        """
        Devuelve el valor actual de un (sensor, variable).
        Lanza KeyError si la combinación nunca ha sido procesada.
        """
        for i in range(len(self._state)):
            item = self._state[i]
            if item[0] == sensor and item[1] == variable:
                return item[2]

        raise KeyError(f"No existe ningún registro procesado para ({sensor}, {variable})")


## 4. Suite de Pruebas Obligatorias
A continuación se detalla la matriz de las 17 pruebas diseñadas para verificar el comportamiento, la integridad del flujo FIFO/LIFO y la gestión de excepciones de la clase

| # | Escenario de Prueba | Operación Ejecutada | Comportamiento / Resultado Esperado |
|---|---|---|---|
| **1** | `pending()` en estado inicial | `p.pending()` | Retorna `0` (cola vacía). |
| **2** | Agregar un registro | `p.add(("S01", "temp", 20))` | Aumenta `pending()` a `1`. |
| **3** | Agregar varios registros | `p.add(...)` tres veces en total | Aumenta `pending()` a `3`. |
| **4** | Verificar procesamiento FIFO | `p.process_next()` | Retorna el primer elemento encolado `("S01", "temp", 20)`. |
| **5** | Procesar un registro | `p.process_next()` | Reduce `pending()` a `2` y actualiza el estado interno. |
| **6** | Procesar varios registros | `p.process_next()` de forma repetida | Reduce `pending()` hasta `0` respetando el orden de llegada. |
| **7** | Actualizar variable existente | Procesar registro con igual `(sensor, variable)` y distinto valor | Sobrescribe el valor en el estado actual sin duplicar la clave. |
| **8** | Consultar valor actual | `p.current_value("S01", "temp")` | Retorna el valor numérico más reciente guardado para dicha clave. |
| **9** | Realizar un `undo()` | `p.undo()` | Revierte únicamente la última modificación realizada por `process_next()` (LIFO). |
| **10** | Múltiples `undo()` consecutivos | `p.undo()` repetidamente | Restaura los estados anteriores progresivamente. |
| **11** | Procesar con cola vacía | `p.process_next()` sin pendientes | Eleva la excepción `Empty` del paquete `goodrich`. |
| **12** | `undo()` con historial vacío | `p.undo()` sin cambios guardados | Eleva la excepción `Empty` del paquete `goodrich`. |
| **13** | Deshacer creación de dato nuevo | `p.undo()` sobre una variable que antes no existía | Elimina la clave del estado actual en lugar de dejarla en `None`. |
| **14** | Múltiples cambios sobre una variable | Procesar varios cambios sobre `(S02, pressure)` y aplicar `undo()` | Recorre el historial en orden inverso recuperando cada valor previo. |
| **15** | Registro con formato incorrecto | `p.add(("S01", "temp"))` | Captura `ValueError` por no tener exactamente 3 elementos. |
| **16** | Registro con valor no numérico | `p.add(("S01", "temp", "texto"))` | Captura `ValueError` al detectar un tipo distinto de `int` o `float`. |
| **17** | Consulta de clave inexistente | `p.current_value("S99", "unknown")` | Eleva `KeyError` al no encontrar la combinación en el estado. |

In [ ]:
# SUITE DE PRUEBAS OBLIGATORIAS (ESCENARIOS 1 AL 17) 

# Instanciación inicial
p = DataProcessor()

# 1. pending() sobre un procesador vacío
print(f"1. Pendientes iniciales: {p.pending()}") # Esperado: 0

# 2. Agregar un registro
p.add(("S01", "temperature", 20.0))
print(f"2. Pendientes tras agregar 1: {p.pending()}") # Esperado: 1

# 3. Agregar varios registros
p.add(("S01", "temperature", 25.0))
p.add(("S01", "humidity", 60.0))
print(f"3. Pendientes tras agregar varios: {p.pending()}") # Esperado: 3

# 4. Verificar procesamiento FIFO
# 5. Procesar un registro
procesado_1 = p.process_next()
print(f"4 y 5. Registro procesado primero (FIFO): {procesado_1}") # Esperado: ("S01", "temperature", 20.0)

# 6. Procesar varios registros
procesado_2 = p.process_next()
procesado_3 = p.process_next()
print(f"6. Procesados siguientes: {procesado_2} y {procesado_3}")

# 7. Actualizar una variable existente (ocurrió al procesar procesado_2)
# 8. Consultar el valor actual
print(f"8. Valor actual S01/temperature: {p.current_value('S01', 'temperature')}") # Esperado: 25.0
print(f"8. Valor actual S01/humidity: {p.current_value('S01', 'humidity')}")       # Esperado: 60.0

# 9. Realizar un undo()
p.undo() # Deshace C (humidity)
print("9. Undo realizado (se eliminó humidity).")

# 10. Realizar varios undo() consecutivos
p.undo() # Deshace B (temperature 25.0)
print(f"10. Valor tras undo() de temperatura: {p.current_value('S01', 'temperature')}") # Esperado: 20.0

p.undo() # Deshace A (temperature 20.0)
print("10. Todos los undo() completados.")

# 11. Procesar cuando la Queue está vacía
try:
    p.process_next()
except Empty as e:
    print(f"11. Error esperado al procesar cola vacía: {e}")

# 12. Hacer undo() cuando el historial está vacío
try:
    p.undo()
except Empty as e:
    print(f"12. Error esperado al hacer undo con historial vacío: {e}")

# 13. Deshacer la creación de un dato que antes no existía
# 14. Hacer varios cambios sobre la misma variable
p.add(("S02", "pressure", 100.0))
p.process_next() # Se crea S02/pressure por 1ra vez
p.add(("S02", "pressure", 105.0))
p.process_next() # Se actualiza a 105.0

p.undo() # Vuelve a 100.0
print(f"14. Valor tras primer undo: {p.current_value('S02', 'pressure')}")

p.undo() # Se elimina completamente S02/pressure (Prueba 13)

# #correccion: antes solo se imprimía un comentario afirmando que el dato
# se eliminó; ahora se verifica explícitamente que current_value() lanza
# KeyError, demostrando (no solo narrando) que la clave ya no existe tras
# deshacer su creación.
try:
    p.current_value("S02", "pressure")
except KeyError as e:
    print(f"13. Confirmado: se deshizo la creación del dato, ya no existe -> {e}")

# 15. Agregar un registro con formato incorrecto
try:
    p.add(("S01", "temperature")) # Solo 2 elementos
except ValueError as e:
    print(f"15. Error capturado (formato incorrecto): {e}")

# 16. Agregar un registro cuyo valor no sea numérico
try:
    p.add(("S01", "temperature", "veinte"))
except ValueError as e:
    print(f"16. Error capturado (valor no numérico): {e}")

# 17. Consultar un sensor/variable que nunca haya sido procesado
try:
    p.current_value("S99", "unknown")
except KeyError as e:
    print(f"17. Error capturado al consultar clave inexistente: {e}")

## 5. Análisis de Complejidad Temporal (Big-O)
A continuación se detalla el análisis de complejidad algorítmica para cada uno de los métodos implementados en la clase **DataProcessor**:

| Método | Complejidad Temporal | Operación Determinante |
|---|---|---|
| `add(record)` | **O(1)** | Operación `enqueue()` en `ArrayQueue` (tiempo constante amortizado). |
| `pending()` | **O(1)** | Consulta directa del tamaño `len()` de la cola. |
| `process_next()` | **O(n)** | Búsqueda lineal `for` sobre la lista `_state` para verificar coincidencia. |
| `undo()` | **O(n)** | Desapilar con `pop()` en `ArrayStack` (**O(1)**) junto con la búsqueda o eliminación en `_state`. |
| `current_value(sensor, variable)` | **O(n)** | Búsqueda secuencial `for` en la lista `_state` hasta encontrar el registro. |

## 6. Explicación de Decisiones de Diseño

En esta sección se justifican las estructuras utilizadas para garantizar un diseño robusto y eficiente en el procesamiento de flujos de datos:

1. **Uso de `ArrayQueue` para Pendientes (FIFO):**
   - Garantiza que las lecturas provenientes de los sensores se procesen exactamente en el mismo orden en el que llegaron.
   - Las operaciones de encolar (`enqueue`) y desencolar (`dequeue`) se realizan en tiempo constante amortizado $O(1)$.

2. **Uso de `ArrayStack` para el Historial (LIFO):**
   - Es la estructura ideal para implementar la funcionalidad de deshacer (`undo`).
   - Mantiene la trazabilidad de las modificaciones en orden inverso a como fueron aplicadas, permitiendo revertir el último cambio procesado en $O(1)$.

3. **Manejo del Estado con `list` y Marca `None`:**
   - La lista `_state` almacena las combinaciones únicas `(sensor, variable, valor)`.
   - Para soportar la reversión de variables nuevas, al procesar por primera vez un registro se guarda en la pila una notita con valor previo `None`. De esta forma, al ejecutar `undo()`, el sistema sabe si debe restaurar un valor anterior o eliminar completamente la clave de la lista.

## 7. Bonus: Implementación del Método redo() (Rehacer)

El método `redo()` permite revertir la acción de un `undo()`, restaurando el cambio previamente deshecho.

### Funcionamiento de la Pila de Rehacer (_redo_stack)

1. **Guardado en undo():** Cuando se deshace un cambio, el estado que se va a eliminar o modificar no se pierde; se guarda en la pila secundaria `_redo_stack`.
2. **Restauración en redo():** Al llamar a `redo()`, se extrae el último cambio de `_redo_stack`, se aplica nuevamente sobre la lista de estado `_state` y se vuelve a registrar en la pila principal `_stack` para permitir futuros `undo()`.
3. **Limpieza por nueva acción:** Si se procesa un nuevo registro mediante `process_next()`, la pila `_redo_stack` se vacía automáticamente para no romper la coherencia del historial.

---

### Análisis del Método redo()

| Propiedad | Descripción |
|---|---|
| **Estructura Interna** | `_redo_stack` (`ArrayStack`) |
| **Comportamiento** | LIFO (Last In, First Out) |
| **Excepción** | Eleva `Empty` si no hay acciones deshechas para rehacer. |
| **Complejidad Temporal** | **O(n)** (debido a la búsqueda en la lista `_state`) |

In [ ]:
# PRUEBA DEL BONUS REDO() 
p_bonus = DataProcessor()

# 1. Agregar y procesar un registro
p_bonus.add(("S01", "temperature", 20.0))
p_bonus.process_next()
print(f"Valor procesado inicial: {p_bonus.current_value('S01', 'temperature')}") # 20.0

# 2. Deshacer el cambio (undo)
p_bonus.undo()
print("Undo realizado.")

# 3. Rehacer el cambio (redo)
p_bonus.redo()
print(f"Valor tras redo(): {p_bonus.current_value('S01', 'temperature')}") # 20.0 (Recuperado)

# 4. Validar excepción cuando no hay nada que rehacer
p_bonus.redo() # Debe lanzar Empty

## 8. Demostración Representativa: Simulación de Monitoreo en Tiempo Real

Para representar el uso de la clase `DataProcessor` en un caso de uso de Ingeniería de Datos, se simula una red de sensores remotos transmitiendo lecturas métricas continuas.

La simulación demuestra:
1. **Consumo progresivo de la cola (`_queue`):** Visualización del drenaje de pendientes mediante el orden FIFO.
2. **Actualización dinámica de la tabla de estado (`_state`):** Reflejo inmediato del valor más reciente por cada par `(sensor, variable)`.
3. **Restauración del sistema ante lecturas anómalas:** Aplicación de `undo()` para remover picos de error y recuperar la estabilidad de la red.

In [ ]:
import time
from IPython.display import clear_output

def monitor_en_tiempo_real():
    procesador = DataProcessor()
    
    # Ráfaga de datos recibida de la red de sensores
    lecturas = [
        ("Sensor_Norte", "Temperatura", 21.5),
        ("Sensor_Norte", "Humedad", 65.0),
        ("Sensor_Sur", "Presion", 1013.2),
        ("Sensor_Norte", "Temperatura", 24.8), # Actualización
        ("Sensor_Sur", "Presion", 1200.0),      # Lectura anómala
    ]
    
    print("Encolando lecturas de la red...")
    for l in lecturas:
        procesador.add(l)
    
    # Procesamiento paso a paso con actualización visual
    while procesador.pending() > 0:
        time.sleep(1)
        clear_output(wait=True)
        
        registro = procesador.process_next()
        
        print("==================================================")
        print("       MONITOR DE DATA STREAM PROCESSOR           ")
        print("==================================================")
        print(f"Último evento procesado (FIFO): {registro}")
        print(f"Lecturas pendientes en cola:    {procesador.pending()}")
        print("--------------------------------------------------")
        print("ESTADO ACTUAL DE LOS SENSORES (_state):")
        for s, v, val in procesador._state:
            print(f"  • [{s}] {v}: {val}")
        print("==================================================\n")

    # Demostración visual de Undo y Redo
    time.sleep(1.5)
    print("¡ALERTA! Presión anómala detectada en Sensor_Sur (1200.0).")
    print("Aplicando rollback de seguridad (undo)...")
    time.sleep(1.5)
    procesador.undo()
    
    clear_output(wait=True)
    print("==================================================")
    print("       ESTADO POST-ROLLBACK (DESPUÉS DE UNDO)      ")
    print("==================================================")
    for s, v, val in procesador._state:
        print(f"  • [{s}] {v}: {val}")
    print("==================================================")

# Ejecutar el monitor
monitor_en_tiempo_real()

## #correccion — Resumen de ajustes aplicados a la entrega original

A partir de la revisión contra la rúbrica del proyecto, se hicieron los siguientes cambios puntuales (ninguno altera el diseño general, que ya era correcto):

1. **Se eliminó la clase `DataProcessor` duplicada.** El notebook original definía la clase dos veces (una versión sin `redo`, y luego la versión final con `redo`); como la segunda pisaba a la primera al ejecutarse, la primera quedaba como código muerto que podía confundir a quien revisara el proyecto. Ahora solo queda la versión definitiva.
2. **Se alineó la documentación en Markdown con el código final:** las secciones 2 y 3 ahora describen los 4 atributos reales de la clase (`_queue`, `_stack`, `_redo_stack`, `_state`), en vez de describir una versión de 3 atributos que ya no correspondía al código ejecutado.
3. **Se reforzó la validación en `add(record)`:** se volvió a incluir `isinstance(record, (tuple, list))` (no solo `len(record) == 3`), para que cualquier tipo de dato "raro" pero indexable con longitud 3 también sea rechazado de forma consistente con `ValueError`, tal como pide la guía. Se cambió también `type(value) != int/float` por `isinstance(value, (int, float))` (excluyendo `bool` explícitamente) por robustez.
4. **Se reforzó la prueba obligatoria #13** (deshacer la creación de un dato que antes no existía): antes solo se imprimía un comentario afirmando el resultado; ahora se verifica explícitamente con `current_value()` envuelto en `try/except KeyError`, igual que se hace en la prueba #17, para que la prueba *demuestre* el resultado en vez de solo narrarlo.

No se modificó la lógica de `process_next()`, `undo()`, `redo()`, `pending()` ni `current_value()`, porque ya cumplían correctamente con la rúbrica.